# Greedy Adversarial Training for IoT Device Identification

This notebook trains 6 models on CSV + JSON datasets with a **3-phase greedy adversarial curriculum**:

| Phase | Epochs | Adversarial Ratio | k_max | Purpose |
|-------|--------|-------------------|-------|---------|
| A | 1-15 | 0% | 0 | Learn the base task on clean data |
| B | 16-30 | 30% | 2 | Gentle introduction of greedy attacks |
| C | 31-50 | 70% | 4 | Full adversarial training against greedy search |

**GreedyAttackSimulator** reads `sensitivity_results.csv` and applies the exact same
perturbations (Zero, Mimic_Mean, Mimic_95th, Padding_x10) on the same vulnerable
features as the attacker. No TRADES or IBP needed — we train against the known attack.

**Models**: LSTM, BiLSTM, CNN-LSTM, XGBoost-LSTM, Transformer, CNN-BiLSTM-Transformer

Each phase saves/loads from **Google Drive** so training can be resumed without re-running.

# Phase 1 : Setup

In [ ]:
# ─── Cell: Setup ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.exists('/content/pfe'):
    !cd /content/pfe && git pull
else:
    !git clone https://github.com/yacinemkk/pfe.git /content/pfe

%cd /content/pfe

!pip install -q torch torchvision tqdm numpy pandas scikit-learn matplotlib xgboost psutil

# Phase 2 : Configuration & Data Pipeline

In [ ]:
# ─── Cell: Configuration ─────────────────────────────────────────────────
import os

JSON_DATA_DIR = '/content/drive/MyDrive/PFE/IPFIX_Records'
CSV_DATA_DIR = '/content/drive/MyDrive/PFE/IPFIX_ML_Instances'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/PFE/results'
DATASETS = 'both'

SEQ_LENGTH = 10
STRIDE = 10
BATCH_SIZE = 256          # Augmenté pour accélérer l'entraînement (optimisé)
LEARNING_RATE = 5e-4
USE_AMP = True           # mixed-precision (fp16) — cuts VRAM ~50%

# Lighter CNN-BiLSTM-Transformer to fit within 22 GB VRAM
CNN_BILSTM_TRANSFORMER_OVERRIDE = {
    'cnn_channels': 32,          # was 64
    'bilstm_hidden': 64,         # was 128  → bilstm output = 128
    'bilstm_layers': 2,
    'bilstm_dropout': 0.3,
    'transformer_d_model': 128,  # was 256
    'transformer_nhead': 4,
    'transformer_layers': 2,
    'transformer_ff_dim': 512,   # reverted to 512
    'transformer_dropout': 0.2,
    'fc_dropout': 0.4,
}

# Greedy adversarial training phases
PHASE_A_EPOCHS = 15
PHASE_B_EPOCHS = 35   # 20 epochs Phase B
PHASE_C_EPOCHS = 55   # 20 epochs Phase C
PHASE_A_MIX_RATIO = 0.0
PHASE_B_MIX_RATIO = 0.3
PHASE_C_MIX_RATIO = 0.7
PHASE_B_K_MAX = 2
PHASE_C_K_MAX = 4

PHASE_D_EPOCHS = 75           # 20 epochs Phase D
PHASE_D_MIX_RATIO = 0.95     # 95% adv, 5% clean anchor
PHASE_D_K_MAX = 4             # k=4 comme Phase C — k=5 était contre-productif

GREEDY_STRATEGIES = ['Zero', 'Mimic_Mean', 'Mimic_95th', 'Padding_x10']

MAX_FILES = None
MAX_RECORDS = None
EVAL_SUBSAMPLE = 1000
EVAL_BATCH_SIZE = 256

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f'CSV data:     {CSV_DATA_DIR}')
print(f'JSON data:    {JSON_DATA_DIR}')
print(f'Results dir:  {DRIVE_RESULTS_DIR}')
print(f'Datasets:     {DATASETS}')
print(f'Seq length:   {SEQ_LENGTH}')
print(f'Phase A: epochs 1-15   | mix=0%   | k_max=0 (clean only)')
print(f'Phase B: epochs 16-30  | mix=30%  | k_max=2')
print(f'Phase C: epochs 31-50 | mix=70%  | k_max=4')
print(f'Batch size:   {BATCH_SIZE}')
print(f'LR:           {LEARNING_RATE}')

import glob
csv_files = glob.glob(f'{CSV_DATA_DIR}/home*_labeled.csv')
print(f'\nFound {len(csv_files)} CSV file(s)')
for f in sorted(csv_files)[:5]:
    size_mb = os.path.getsize(f) / (1024**2)
    print(f'  {os.path.basename(f)} ({size_mb:.1f} MB)')
if len(csv_files) > 5:
    print(f'  ... and {len(csv_files) - 5} more')

json_files = glob.glob(f'{JSON_DATA_DIR}/**/*.json', recursive=True)
print(f'\nFound {len(json_files)} JSON file(s)')
for f in json_files:
    size_gb = os.path.getsize(f) / (1024**3)
    print(f'  {os.path.basename(f)} ({size_gb:.1f} GB)')



In [ ]:
# ─── Cell: RAM Monitoring & Data Loading ─────────────────────────────────
import gc
import psutil
import torch
import os
import numpy as np
import pickle
import glob

def get_memory_usage():
    process = psutil.Process(os.getpid())
    ram_gb = process.memory_info().rss / (1024**3)
    gpu_gb = 0
    if torch.cuda.is_available():
        gpu_gb = torch.cuda.memory_allocated() / (1024**3)
    return ram_gb, gpu_gb

def log_memory(label=''):
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [RAM {label}] {ram_gb:.2f} GB | [GPU {label}] {gpu_gb:.2f} GB')

def aggressive_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    ram_gb, gpu_gb = get_memory_usage()
    print(f'  [Cleanup] RAM: {ram_gb:.2f} GB | GPU: {gpu_gb:.2f} GB')

print('RAM monitoring utilities loaded.')
log_memory('startup')


def load_and_display_csv_dataset(csv_data_dir, seq_length=10, stride=10, save_dir=None):
    import sys
    sys.path.insert(0, '/content/pfe')
    from src.data.preprocessor import IoTDataProcessor

    print('\n' + '=' * 70)
    print('  LOADING CSV DATASET')
    print('=' * 70)

    processor = IoTDataProcessor()
    result = processor.process_all(
        max_files=None,
        data_dir=csv_data_dir,
        seq_length=seq_length,
        stride=stride,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = len(features)

    print(f'  Features ({n_continuous}): {features[:5]}...')
    print(f'  Classes ({len(label_encoder.classes_)}): {list(label_encoder.classes_)}')
    print(f'  Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f'  Saving preprocessed CSV to Drive...')
        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)
        with open(f'{save_dir}/csv_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features, 'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length, 'stride': stride,
            }, f)
        with open(f'{save_dir}/csv_ready', 'w') as f:
            f.write('ready')
        print(f'  CSV dataset saved to Drive.')

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }


def load_and_display_json_dataset(json_data_dir, seq_length=10, stride=10, max_records=None, save_dir=None):
    import sys
    sys.path.insert(0, '/content/pfe')
    from src.data.json_preprocessor import JsonIoTDataProcessor

    print('\n' + '=' * 70)
    print('  LOADING JSON DATASET')
    print('=' * 70)

    processor = JsonIoTDataProcessor()
    result = processor.process_all(
        data_dir=json_data_dir,
        seq_length=seq_length,
        stride=stride,
        max_records=max_records,
        apply_balancing=False,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = 36

    print(f'  Features ({len(features)}): {features[:5]}...')
    print(f'  Classes ({len(label_encoder.classes_)}): {list(label_encoder.classes_)}')
    print(f'  Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f'  Saving preprocessed JSON to Drive...')
        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)
        with open(f'{save_dir}/json_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features, 'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length, 'stride': stride,
            }, f)
        with open(f'{save_dir}/json_ready', 'w') as f:
            f.write('ready')
        print(f'  JSON dataset saved to Drive.')

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }


def load_dataset_from_drive(dataset_type):
    preprocessed_dir = f'{DRIVE_RESULTS_DIR}/preprocessed/{dataset_type}'
    ready_file = f'{preprocessed_dir}/{dataset_type}_ready'

    if os.path.exists(ready_file) and os.path.exists(f'{preprocessed_dir}/X_train.npy'):
        print(f'  Loading preprocessed {dataset_type.upper()} from Drive...')
        X_train = np.load(f'{preprocessed_dir}/X_train.npy')
        X_val = np.load(f'{preprocessed_dir}/X_val.npy')
        X_test = np.load(f'{preprocessed_dir}/X_test.npy')
        y_train = np.load(f'{preprocessed_dir}/y_train.npy')
        y_val = np.load(f'{preprocessed_dir}/y_val.npy')
        y_test = np.load(f'{preprocessed_dir}/y_test.npy')

        meta_file = f'{preprocessed_dir}/{dataset_type}_metadata.pkl'
        with open(meta_file, 'rb') as f:
            metadata = pickle.load(f)

        data = {
            'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
            'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
            'features': metadata['features'], 'scaler': metadata['scaler'],
            'label_encoder': metadata['label_encoder'], 'n_continuous': metadata['n_continuous']
        }
        print(f'  Preprocessed {dataset_type.upper()} loaded: {len(X_train):,} train samples')
        return data
    else:
        print(f'  No preprocessed data found. Loading {dataset_type.upper()} fresh...')
        if dataset_type == 'csv':
            return load_and_display_csv_dataset(
                CSV_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
                save_dir=CSV_PREPROCESSED_DIR
            )
        else:
            return load_and_display_json_dataset(
                JSON_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
                max_records=MAX_RECORDS, save_dir=JSON_PREPROCESSED_DIR
            )

CSV_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/csv'
JSON_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/json'

print('Data loading functions ready.')

In [ ]:
def load_and_display_csv_dataset(csv_data_dir, seq_length=10, stride=10, save_dir=None):
    """Charge COMPLÈTEMENT le dataset CSV, affiche les infos, et sauvegarde sur Drive."""
    import sys
    import gc
    import numpy as np
    import pickle

    sys.path.insert(0, '/content/pfe')

    print("\n" + "=" * 70)
    print("  CHARGEMENT COMPLET DU DATASET CSV")
    print("=" * 70)

    print(f"\n  Répertoire : {csv_data_dir}")
    print(f"  Seq length : {seq_length} | Stride : {stride}")

    csv_files = sorted(glob.glob(f'{csv_data_dir}/home*_labeled.csv'))
    print(f"\n  Fichiers CSV trouvés : {len(csv_files)}")

    for f in csv_files:
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"    {os.path.basename(f):<30s} : {size_mb:>10.1f} MB")

    total_gb = sum(os.path.getsize(f) for f in csv_files) / (1024**3)
    print(f"\n  Taille totale : {total_gb:.2f} GB")

    # Charger le dataset complet via le vrai pipeline CSV
    print("\n  Chargement via le pipeline CSV (IoTDataProcessor)...")
    from src.data.preprocessor import IoTDataProcessor

    processor = IoTDataProcessor()
    result = processor.process_all(
        max_files=None,
        data_dir=csv_data_dir,
        seq_length=seq_length,
        stride=stride,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = len(features)

    print(f"\n  {'='*70}")
    print(f"  RÉSULTAT DU CHARGEMENT")
    print(f"  {'='*70}")
    print(f"    Features ({n_continuous}) : {features[:5]}...")
    print(f"    Classes ({len(label_encoder.classes_)}) : {list(label_encoder.classes_)}")
    print(f"\n  Shapes des séquences (seq_length={seq_length}, stride={stride}) :")
    print(f"    Train : {X_train.shape}  →  {len(X_train):,} séquences")
    print(f"    Val   : {X_val.shape}  →  {len(X_val):,} séquences")
    print(f"    Test  : {X_test.shape}  →  {len(X_test):,} séquences")
    print(f"    Total : {len(X_train) + len(X_val) + len(X_test):,} séquences")

    print(f"\n  Distribution des classes (train) :")
    for cls in label_encoder.classes_:
        cls_id = label_encoder.transform([cls])[0]
        count = int(np.sum(y_train == cls_id))
        bar = '█' * max(1, count // 50)
        print(f"    {cls:<30s} : {count:>6,}  {bar}")

    # ─── Sauvegarder sur Drive ────────────────────────────────────────────
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f"\n  💾 Sauvegarde du dataset CSV pré-traité sur Drive...")
        print(f"     Répertoire : {save_dir}")

        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)

        with open(f'{save_dir}/csv_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features,
                'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length,
                'stride': stride,
            }, f)

        # Marker file to indicate preprocessing is complete
        with open(f'{save_dir}/csv_ready', 'w') as f:
            f.write('ready')

        saved_gb = (X_train.nbytes + X_val.nbytes + X_test.nbytes +
                    y_train.nbytes + y_val.nbytes + y_test.nbytes) / (1024**3)
        print(f"  ✅ Dataset CSV sauvegardé ({saved_gb:.2f} GB)")
        print(f"     Fichiers : X_train, X_val, X_test, y_train, y_val, y_test, csv_metadata.pkl")

    print(f"\n  {'='*70}")
    print(f"  ✅ Dataset CSV chargé complètement en RAM")
    print(f"  {'='*70}\n")

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

# ─── CSV preprocessing directory on Drive ─────────────────────────────
CSV_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/csv'

if DATASETS in ['csv', 'both']:
    csv_data = load_and_display_csv_dataset(
        CSV_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
        save_dir=CSV_PREPROCESSED_DIR
    )
else:
    csv_data = None
    print('Skipping CSV dataset loading — DATASETS is not csv or both')


## CSV Dataset Loading

In [ ]:
# ─── Cell: Load CSV Dataset ──────────────────────────────────────────────
if DATASETS in ['csv', 'both']:
    csv_data = load_dataset_from_drive('csv')
else:
    csv_data = None
    print('Skipping CSV dataset')

# Phase 3 : Greedy Adversarial Training Core

In [ ]:
# ─── Cell: GreedyAttackSimulator + Training Functions ─────────────────────
import sys
sys.path.insert(0, '/content/pfe')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import json
import os

from src.models.lstm import LSTMClassifier
from src.models.bilstm import BiLSTMClassifier
from src.models.cnn_lstm import CNNLSTMClassifier
from src.models.xgboost_lstm import XGBoostLSTMClassifier
from src.models.transformer import TransformerClassifier
from src.models.cnn_bilstm_transformer import CNNBiLSTMTransformerClassifier
from src.training.trainer import IoTSequenceDataset
from src.adversarial.robust_losses import AFDLoss


# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic
# GreedyAttackSimulator — replique exactement adversarial_search_seq.py
# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic

class GreedyAttackSimulator:
    def __init__(self, sensitivity_results, feature_stats, feature_names=None,
                 n_continuous=None, verbose=True):
        self.results = sensitivity_results
        self.stats = feature_stats
        self.feature_names = feature_names or []
        self.n_continuous = n_continuous
        
        self.feature_pool = {}
        self.feature_weights = {}
        epsilon = 0.05  # Exploratory minimum probability
        
        for fi, st, drop in sensitivity_results:
            if fi not in self.feature_pool:
                self.feature_pool[fi] = []
                self.feature_weights[fi] = max(0.0, drop) + epsilon
            
            # Maintain a pool of valid strategies even if drop <= 0, we keep them for exploratory testing
            self.feature_pool[fi].append(st)
                
        self.available_features = list(self.feature_pool.keys())
        if len(self.available_features) > 0:
            weights = np.array([self.feature_weights[f] for f in self.available_features])
            if weights.sum() > 0:
                self.sampling_probs = weights / weights.sum()
            else:
                self.sampling_probs = np.ones(len(weights)) / len(weights)
        else:
            self.sampling_probs = np.array([])

        # ─── Contraintes de réalisme (projection) ─────────────────────────
        self._build_constraints()

        if verbose:
            print(f"  [Simulator] Vulnerability Dictionary created with {len(self.available_features)} distinct features.")
            print(f"  [Simulator] Projection: {len(self.dependent_indices)} dependent pairs, "
                  f"n_continuous={self.n_continuous}")
            print(f"  [Simulator] Top 3 features logic overview:")
            for idx, feat in enumerate(self.available_features[:3]):
                print(f"     -> Feature {feat} mapped to {len(self.feature_pool[feat])} strategies (prob={self.sampling_probs[idx]:.3f})")

    def _build_constraints(self):
        """Build non-modifiable list and dependent pairs from feature names."""
        fnames = self.feature_names
        has_pkt_dir = any(f.startswith('pkt_dir_') for f in fnames)

        if has_pkt_dir:
            self.dependent_pairs = {
                'reversePacketTotalCount': 'packetTotalCount',
                'reverseOctetTotalCount': 'octetTotalCount',
                'reverseAverageInterarrivalTime': 'averageInterarrivalTime',
            }
        else:
            self.dependent_pairs = {
                'inPacketCount': 'outPacketCount',
                'inByteCount': 'outByteCount',
                'inAvgIAT': 'outAvgIAT',
                'inAvgPacketSize': 'outAvgPacketSize',
            }

        # Build index pairs (indep_idx, dep_idx)
        self.dependent_indices = []
        for dep_name, indep_name in self.dependent_pairs.items():
            if dep_name in fnames and indep_name in fnames:
                self.dependent_indices.append(
                    (fnames.index(indep_name), fnames.index(dep_name))
                )

    def projection(self, X):
        """Clip perturbed values to valid ranges and enforce dependent constraints."""
        X_proj = X.copy()
        n_cont = self.n_continuous

        # Clip continuous features to [-3, 3] and categorical to {0, 1}
        if n_cont is not None:
            if X_proj.ndim == 3:
                X_proj[:, :, :n_cont] = np.clip(X_proj[:, :, :n_cont], -3.0, 3.0)
                X_proj[:, :, n_cont:] = np.clip(np.round(X_proj[:, :, n_cont:]), 0, 1)
            elif X_proj.ndim == 2:
                X_proj[:, :n_cont] = np.clip(X_proj[:, :n_cont], -3.0, 3.0)
                X_proj[:, n_cont:] = np.clip(np.round(X_proj[:, n_cont:]), 0, 1)
        else:
            X_proj = np.clip(X_proj, -3.0, 3.0)

        # Enforce dependent feature correlations (ratio clamped to [0.5, 2.0])
        for indep_idx, dep_idx in self.dependent_indices:
            if n_cont is not None and (dep_idx >= n_cont or indep_idx >= n_cont):
                continue
            if X_proj.ndim == 3:
                ratio = np.abs(X_proj[:, :, dep_idx]) / (
                    np.abs(X_proj[:, :, indep_idx]) + 1e-8
                )
                X_proj[:, :, dep_idx] = X_proj[:, :, indep_idx] * np.clip(ratio, 0.5, 2.0)
            elif X_proj.ndim == 2:
                ratio = np.abs(X_proj[:, dep_idx]) / (
                    np.abs(X_proj[:, indep_idx]) + 1e-8
                )
                X_proj[:, dep_idx] = X_proj[:, indep_idx] * np.clip(ratio, 0.5, 2.0)

        return X_proj

    def save_dictionary(self, save_path, feature_names):
        dict_data = {
            "num_features": len(self.available_features),
            "features": {}
        }
        for feat_idx in self.available_features:
            feat_name = feature_names[feat_idx] if feat_idx < len(feature_names) else f"f{feat_idx}"
            dict_data["features"][feat_name] = {
                "strategies": self.feature_pool[feat_idx],
                "weight": float(self.feature_weights[feat_idx])
            }
        import os
        import json
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, 'w') as f:
            json.dump(dict_data, f, indent=2)
        print(f"  [Simulator] Vulnerability Dictionary saved to {save_path}")

    @classmethod
    def compute_feature_stats(cls, X_train):
        X_flat = X_train.reshape(-1, X_train.shape[-1])
        stats = {}
        for i in range(X_flat.shape[1]):
            col = X_flat[:, i]
            stats[i] = {
                'mean': float(col.mean()),
                'p95': float(np.percentile(col, 95)),
                'std': float(col.std()),
            }
        return stats

    def apply_strategy(self, X, feat_idx, strategy):
        X = X.copy()
        if strategy == 'Zero':
            X[:, :, feat_idx] = 0.0
        elif strategy == 'Mimic_Mean':
            X[:, :, feat_idx] = self.stats[feat_idx]['mean']
        elif strategy == 'Mimic_95th':
            X[:, :, feat_idx] = self.stats[feat_idx]['p95']
        elif strategy == 'Padding_x10':
            X[:, :, feat_idx] = np.clip(X[:, :, feat_idx] * 10.0, -5.0, 5.0)
        return X

    def generate_greedy(self, X, k):
        """Méthode stochastique originale — sélection aléatoire pondérée de k features."""
        X_adv = X.copy()
        n_avail = len(self.available_features)
        if n_avail == 0:
            return X_adv

        k_actual = min(k, n_avail)
        chosen_features = np.random.choice(self.available_features, size=k_actual, replace=False, p=self.sampling_probs)

        for feat_idx in chosen_features:
            strategy = np.random.choice(self.feature_pool[feat_idx])
            X_adv = self.apply_strategy(X_adv, feat_idx, strategy)

        X_adv = self.projection(X_adv)
        return X_adv

    # ══════════════════════════════════════════════════════════════════════
    # MÉTHODE 3 — Stratified All-K Training (curriculum exhaustif par niveau)
    # Phase B : exhaustif k=1   → 100% couverture des attaques single-feature
    # Phase C : exhaustif k=1+2 → couverture des paires critiques
    # Phase D : exhaustif k=1+2 + worst-case k≥3 (Méthode 2, PGD-like)
    # ══════════════════════════════════════════════════════════════════════

    def generate_all_k1(self, X):
        """Génère TOUTES les attaques k=1 pour le batch X.

        Pour chaque paire (feature, strategy) du dictionnaire de vulnérabilités,
        crée une version adversariale du batch complet.
        Retourne (X_augmented, n_attacks) où X_augmented a n*(1+n_attacks) lignes
        (clean en premier, puis une copie par attaque).
        """
        all_X = []  # 100% adversarial — pas de copie clean
        for feat_idx in self.available_features:
            for strategy in self.feature_pool[feat_idx]:
                X_adv = self.apply_strategy(X.copy(), feat_idx, strategy)
                X_adv = self.projection(X_adv)
                all_X.append(X_adv)
        n_attacks = len(all_X)  # tous les elements sont adversariaux
        return np.concatenate(all_X, axis=0), n_attacks

    def generate_all_k2(self, X, top_n=8):
        """Génère toutes les attaques k=1 + les paires k=2 des top-N features.

        Pour les attaques k=2, utilise les top_n features les plus vulnérables
        (selon le dictionnaire trié par poids/sensibilité) avec leur stratégie
        la plus efficace (première du pool). Couvre les interactions de paires
        les plus dangereuses sans explosion combinatoire.
        """
        all_X = []  # 100% adversarial — pas de copie clean
        # k=1 : exhaustif sur tout le dictionnaire
        for feat_idx in self.available_features:
            for strategy in self.feature_pool[feat_idx]:
                X_adv = self.apply_strategy(X.copy(), feat_idx, strategy)
                all_X.append(self.projection(X_adv))
        # k=2 : paires des top-N features (meilleure stratégie de chacun)
        top_feats = self.available_features[:min(top_n, len(self.available_features))]
        for i, f1 in enumerate(top_feats):
            s1 = self.feature_pool[f1][0]
            for f2 in top_feats[i + 1:]:
                s2 = self.feature_pool[f2][0]
                X_adv = self.apply_strategy(X.copy(), f1, s1)
                X_adv = self.apply_strategy(X_adv, f2, s2)
                all_X.append(self.projection(X_adv))
        n_attacks = len(all_X)  # tous les elements sont adversariaux
        return np.concatenate(all_X, axis=0), n_attacks

    def generate_worst_case_k(self, X, y_np, model, device, k=3, n_candidates=8):
        """Sélection worst-case vectorisée parmi n_candidates attaques pour k features (Phase D).
        
        Optimisation: Génère l'attaque stochastique sur l'ensemble du batch simultanément
        et fait l'inférence en une seule passe sur le GPU. Vitesse x100 par rapport à l'original.
        """
        model.eval()
        N = len(X)
        best_X = X.copy()
        best_losses = np.full(N, -np.inf)
        
        y_t = torch.LongTensor(y_np).to(device)
        
        with torch.no_grad():
            for _ in range(n_candidates):
                # Génération pour tout le batch d'un coup (extrêmement rapide)
                X_cand = self.generate_greedy(X, k)
                X_t = torch.FloatTensor(X_cand).to(device)
                
                # Inférence massive GPU
                logits = model(X_t)
                
                # Pertes individuelles
                losses = F.cross_entropy(logits, y_t, reduction='none').cpu().numpy()
                
                # Mise à jour des meilleurs candidats
                mask = losses > best_losses
                best_losses[mask] = losses[mask]
                best_X[mask] = X_cand[mask]

        model.train()
        return best_X
    def generate_training_batch_stratified(self, X, y_np, model, device,
                                            phase='B', k_max=4,
                                            mix_ratio=0.5, n_candidates=8,
                                            top_n_k2=8):
        """Génération stratifiée selon la phase du curriculum (Méthode 3).

        Phase B → generate_all_k1     : 100% des attaques single-feature
        Phase C → generate_all_k2     : k=1 exhaustif + paires k=2 top-N features
        Phase D → generate_all_k2     : k=1+k=2 exhaustif
                + generate_worst_case_k pour k=3 et k=4 (worst-case / PGD-like)

        Retourne (X_out, y_out) avec les labels dupliqués en correspondance.
        """
        # ── Ancrage de classe progressif : split clean/adv selon mix_ratio ──
        n_adv_split = int(len(X) * mix_ratio)
        n_clean_split = len(X) - n_adv_split
        X_clean_anchor = X[:n_clean_split]
        y_clean_anchor = y_np[:n_clean_split]
        X_for_atk = X[n_clean_split:] if n_adv_split > 0 else X[:0]
        y_for_atk = y_np[n_clean_split:] if n_adv_split > 0 else y_np[:0]

        if phase == 'B':
            if len(X_for_atk) > 0:
                X_atk, n_atk = self.generate_all_k1(X_for_atk)
                y_atk = np.tile(y_for_atk, n_atk)[:len(X_atk)]
            else:
                X_atk, y_atk = X_for_atk, y_for_atk
                n_atk = 0
            if not getattr(self, '_logged_b', False):
                print(f"  [Stratified-B] {n_clean_split} clean + {n_atk} attaques k=1 × {len(X_for_atk)} samples → batch {n_clean_split + len(X_atk)}")
                self._logged_b = True
            return (np.concatenate([X_clean_anchor, X_atk], axis=0),
                    np.concatenate([y_clean_anchor, y_atk], axis=0))

        elif phase == 'C':
            if len(X_for_atk) > 0:
                X_atk, n_atk = self.generate_all_k2(X_for_atk, top_n=top_n_k2)
                y_atk = np.tile(y_for_atk, n_atk)[:len(X_atk)]
            else:
                X_atk, y_atk = X_for_atk, y_for_atk
                n_atk = 0
            if not getattr(self, '_logged_c', False):
                print(f"  [Stratified-C] {n_clean_split} clean + {n_atk} attaques k=1+k=2 × {len(X_for_atk)} samples → batch {n_clean_split + len(X_atk)}")
                self._logged_c = True
            return (np.concatenate([X_clean_anchor, X_atk], axis=0),
                    np.concatenate([y_clean_anchor, y_atk], axis=0))

        elif phase == 'D':
            parts_X = [X_clean_anchor]
            parts_y = [y_clean_anchor]
            if len(X_for_atk) > 0:
                X_exh, n_exh = self.generate_all_k2(X_for_atk, top_n=top_n_k2)
                y_exh = np.tile(y_for_atk, n_exh)[:len(X_exh)]
                parts_X.append(X_exh)
                parts_y.append(y_exh)
                for k_val in range(3, k_max + 1):
                    X_wc = self.generate_worst_case_k(
                        X_for_atk, y_for_atk, model, device, k=k_val, n_candidates=n_candidates
                    )
                    parts_X.append(X_wc)
                    parts_y.append(y_for_atk.copy())
            if not getattr(self, '_logged_d', False):
                total = sum(len(p) for p in parts_X)
                print(f"  [Stratified-D] {n_clean_split} clean + worst-case k=1..4: batch total {total} samples")
                self._logged_d = True
            return np.concatenate(parts_X, axis=0), np.concatenate(parts_y, axis=0)

        else:
            # Fallback stochastique (Phase A ou autre)
            X_out = X.copy()
            n_adv = int(len(X) * mix_ratio)
            idx_adv = np.random.choice(len(X), n_adv, replace=False)
            for i in idx_adv:
                k = np.random.randint(1, k_max + 1)
                X_out[[i]] = self.generate_greedy(X[[i]], k)
            return X_out, y_np.copy()

    def generate_training_batch(self, X, k_max=4, mix_ratio=0.5):
        """Méthode stochastique originale — conservée pour Phase A et compatibilité."""
        n = len(X)
        n_adv = int(n * mix_ratio)
        idx_adv = np.random.choice(n, n_adv, replace=False)
        X_out = X.copy()
        flags = np.zeros(n, dtype=np.float32)
        for i in idx_adv:
            k = np.random.randint(1, k_max + 1)
            X_out[[i]] = self.generate_greedy(X[[i]], k)
            flags[i] = 1.0
        return X_out, flags

def load_sensitivity_results(csv_path, feature_names):
    df = pd.read_csv(csv_path).sort_values('drop', ascending=False)
    idx = {name: i for i, name in enumerate(feature_names)}
    result = []
    for _, row in df.iterrows():
        feat = row['feature']
        if feat in idx:
            result.append((idx[feat], row['strategy'], float(row['drop'])))
    print(f"  -> {len(result)} (feature, strategy) pairs loaded from sensitivity analysis")
    print(f"  Top 5 most vulnerable:")
    for i, (fi, st, dr) in enumerate(result[:5], 1):
        print(f"     {i}. {feature_names[fi]:<25} | {st:<14} | drop={dr*100:.1f}%")
    return result


def create_model(model_type, input_size, num_classes):
    if model_type == 'lstm':
        return LSTMClassifier(input_size, num_classes)
    elif model_type == 'bilstm':
        return BiLSTMClassifier(input_size, num_classes)
    elif model_type == 'cnn_lstm':
        return CNNLSTMClassifier(input_size, num_classes)
    elif model_type == 'xgboost_lstm':
        return XGBoostLSTMClassifier(input_size, num_classes)
    elif model_type == 'transformer':
        return TransformerClassifier(input_size, num_classes)
    elif model_type == 'cnn_bilstm_transformer':
        return CNNBiLSTMTransformerClassifier(input_size, num_classes, seq_length=SEQ_LENGTH,
                                              config=CNN_BILSTM_TRANSFORMER_OVERRIDE)
    elif model_type == 'nlp_cnn_bilstm_transformer':
        return CNNBiLSTMTransformerClassifier(input_size=128, num_classes=num_classes, seq_length=576, vocab_size=52000, config=CNN_BILSTM_TRANSFORMER_OVERRIDE)
    else:
        raise ValueError(f"Unknown model type: {model_type}")


# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic
# train_greedy_phase — train model for one phase (A/B/C)
# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic

def train_greedy_phase(
    model, X_train, y_train, X_val, y_val,
    phase, start_epoch, end_epoch,
    mix_ratio, k_max,
    p_drop=0.0, sigma_noise=0.0, afd_lambda=0.0,
    simulator=None, device=None,
    lr=5e-4, batch_size=64, save_path=None,
    is_nlp=False, tokenizer=None, features=None,
    adv_method='stochastic', n_candidates=8, top_n_k2=8,
):
    # adv_method: 'stochastic'  → méthode originale (Phase A)
    #             'stratified'  → Méthode 3 curriculum exhaustif
    #               Phase B: exhaustif k=1
    #               Phase C: exhaustif k=1 + k=2 top-8 paires
    #               Phase D: exhaustif k=1+k=2 + worst-case k=3,4 (PGD-like)
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    n_epochs_phase = end_epoch - start_epoch + 1
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(n_epochs_phase, 1), eta_min=1e-6
    )
    use_amp = USE_AMP and device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    if start_epoch > 1:
        for _ in range(start_epoch - 1):
            optimizer.step()
            scheduler.step()

    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)

    phase_names = {'A': 'Fondation (clean only)', 'B': 'Introduction (30% adv, k_max=2)', 'C': 'Principal (70% adv, k_max=4)', 'D': 'Consolidation (85% adv, k_max=4, epochs 51-80)'}
    adv_method_desc = {
        'stochastic':  'Stochastique (original) — k features tirées aléatoirement',
        'stratified':  {
            'A': 'Stochastique (Phase A — clean only)',
            'B': 'STRATIFIED k=1 exhaustif — 100% couverture single-feature',
            'C': 'STRATIFIED k=1+k=2 exhaustif — paires top-8 features',
            'D': 'STRATIFIED k=1+k=2 exhaustif + WORST-CASE k=3,4 (PGD-like)',
        },
    }
    if adv_method == 'stratified':
        method_label = adv_method_desc['stratified'].get(phase, 'stratified')
    else:
        method_label = adv_method_desc.get(adv_method, adv_method)

    print(f"\n{'='*60}")
    print(f"  PHASE {phase} — epochs {start_epoch}-{end_epoch}")
    print(f"  {phase_names.get(phase, '')}")
    print(f"  mix_ratio={mix_ratio} | k_max={k_max}")
    print(f"  ADV METHOD : {method_label}")
    print(f"{'='*60}")

    best_val_acc = 0.0
    best_combined = 0.0   # score = 0.4*clean + 0.6*adv (phases adv seulement)
    best_epoch = start_epoch

    epoch_dir = save_path.replace('.pt', '_epochs') if save_path else None
    if epoch_dir and os.path.exists(epoch_dir):
        saved_files = [f for f in os.listdir(epoch_dir) if f.startswith('epoch_') and f.endswith('.pt')]
        if saved_files:
            epochs_present = [int(f.replace('epoch_', '').replace('.pt', '')) for f in saved_files]
            last_saved = max(epochs_present)
            if start_epoch <= last_saved < end_epoch:
                print(f"  [Resumption] Reprise de l'entraînement à partir de l'époque {last_saved+1}...")
                ckpt = torch.load(f"{epoch_dir}/epoch_{last_saved}.pt", map_location=device)
                model.load_state_dict(ckpt['model_state_dict'])
                best_val_acc = ckpt.get('best_val_acc', 0.0)
                best_combined = ckpt.get('best_combined', 0.0)
                best_epoch = ckpt.get('best_epoch', start_epoch)
                start_epoch = last_saved + 1

    label_sm_map = {'A': 0.05, 'B': 0.08, 'C': 0.10}
    label_sm = label_sm_map.get(phase, 0.05)

    for epoch in range(start_epoch, end_epoch + 1):
        model.train()
        criterion = nn.CrossEntropyLoss(label_smoothing=label_sm)

        total_loss, total_correct, total_n = 0.0, 0, 0

        if afd_lambda > 0:
            num_classes = len(np.unique(y_train))
            afd_criterion = AFDLoss(num_classes, num_classes, lambda_intra=1.0, lambda_inter=0.5).to(device)

        for batch_idx, (X_batch, y_batch) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)):
            X_np = X_batch.numpy()
            y_input = y_batch.to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=use_amp):
                if mix_ratio > 0 and simulator is not None:
                    if afd_lambda > 0:
                        X_clean_t = X_batch.to(device)
                        X_adv_mixed, _ = simulator.generate_training_batch(X_np, k_max=k_max, mix_ratio=mix_ratio)
                        X_adv_t = torch.FloatTensor(X_adv_mixed).to(device)

                        if is_nlp:
                            X_clean_t = torch.LongTensor(tokenizer.transform(X_batch.numpy(), features)).to(device)
                            X_adv_t = torch.LongTensor(tokenizer.transform(X_adv_mixed, features)).to(device)

                        if sigma_noise > 0:
                            X_clean_t = X_clean_t + torch.randn_like(X_clean_t) * sigma_noise
                            X_adv_t = X_adv_t + torch.randn_like(X_adv_t) * sigma_noise

                        logits_clean = model(X_clean_t)
                        logits_adv = model(X_adv_t)

                        loss_ce = criterion(logits_adv, y_input)
                        loss_afd = afd_criterion(logits_clean, logits_adv, y_input)
                        loss = loss_ce + afd_lambda * loss_afd
                        logits = logits_adv
                    else:
                        # ── Génération adversariale stratifiée ou stochastique ───────────
                        if adv_method == 'stratified' and phase in ('B', 'C', 'D'):
                            y_np_batch = y_batch.numpy()
                            X_mixed, y_mixed_np = simulator.generate_training_batch_stratified(
                                X_np, y_np_batch, model, device,
                                phase=phase, k_max=k_max, mix_ratio=mix_ratio,
                                n_candidates=n_candidates, top_n_k2=top_n_k2,
                            )
                            y_input = torch.LongTensor(y_mixed_np).to(device)
                        else:
                            X_mixed, _ = simulator.generate_training_batch(X_np, k_max=k_max, mix_ratio=mix_ratio)

                        if is_nlp:
                            X_input = torch.LongTensor(tokenizer.transform(X_mixed, features)).to(device)
                        else:
                            X_input = torch.FloatTensor(X_mixed).to(device)
                        if p_drop > 0 and not is_nlp:
                            mask = (torch.rand(X_input.shape[0], 1, X_input.shape[2], device=device) > p_drop).float()
                            X_input = X_input * mask / (1.0 - p_drop)
                        if sigma_noise > 0:
                            X_input = X_input + torch.randn_like(X_input) * sigma_noise

                        logits = model(X_input)
                        loss = criterion(logits, y_input)
                else:
                    if is_nlp:
                        X_input = torch.LongTensor(tokenizer.transform(X_np, features)).to(device)
                    else:
                        X_input = X_batch.to(device)
                    if p_drop > 0 and not is_nlp:
                        mask = (torch.rand(X_input.shape[0], 1, X_input.shape[2], device=device) > p_drop).float()
                        X_input = X_input * mask / (1.0 - p_drop)
                    if sigma_noise > 0:
                        X_input = X_input + torch.randn_like(X_input) * sigma_noise



                    logits = model(X_input)
                    loss = criterion(logits, y_input)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * len(y_input)
            total_correct += (logits.argmax(1) == y_input).sum().item()
            total_n += len(y_input)

        scheduler.step()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        train_loss = total_loss / total_n
        train_acc = total_correct / total_n

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for i in range(0, len(X_val), batch_size):
                end = min(i + batch_size, len(X_val))
                if is_nlp:
                    X_val_t = torch.LongTensor(tokenizer.transform(X_val[i:end], features)).to(device)
                else:
                    X_val_t = torch.FloatTensor(X_val[i:end]).to(device)
                y_val_t = torch.LongTensor(y_val[i:end]).to(device)
                val_correct += (model(X_val_t).argmax(1) == y_val_t).sum().item()
                val_total += len(y_val_t)
            val_clean_acc = val_correct / val_total

            val_adv_acc = 0.0
            val_adv_acc_k = {}
            if simulator is not None and k_max > 0:
                n_eval = min(EVAL_SUBSAMPLE, len(X_val))
                np.random.seed(42)
                torch.manual_seed(42)
                for k_val in range(1, k_max + 1):
                    val_adv_correct = 0
                    for i in range(0, n_eval, batch_size):
                        end = min(i + batch_size, n_eval)
                        X_adv_np = simulator.generate_greedy(X_val[i:end], k=k_val)
                        if is_nlp:
                            X_adv_t = torch.LongTensor(tokenizer.transform(X_adv_np, features)).to(device)
                        else:
                            X_adv_t = torch.FloatTensor(X_adv_np).to(device)
                        y_sub = torch.LongTensor(y_val[i:end]).to(device)
                        val_adv_correct += (model(X_adv_t).argmax(1) == y_sub).sum().item()
                    val_adv_acc_k[k_val] = val_adv_correct / n_eval
                val_adv_acc = val_adv_acc_k[k_max]

        adv_str = " ".join([f"k{k}={acc:.4f}" for k, acc in val_adv_acc_k.items()]) if simulator else ""
        print(f"  Epoch {epoch:3d}/{end_epoch} [Ph{phase}] "
              f"Loss={train_loss:.4f} TrainAcc={train_acc:.4f} "
              f"CleanAcc={val_clean_acc:.4f} AdvAcc={val_adv_acc:.4f}  {adv_str}")

        # Phase A : sélection sur clean seulement (pas encore d'attaques)
        # Phases B/C/D : score combiné → favorise la robustesse adversariale
        if phase == 'A':
            selection_score = val_clean_acc
            is_better = val_clean_acc > best_val_acc
        else:
            # 0.4 clean + 0.6 adv : on cherche ~85% adv sans effondrer le clean
            selection_score = 0.4 * val_clean_acc + 0.6 * val_adv_acc
            is_better = selection_score > best_combined

        if is_better:
            best_val_acc = val_clean_acc
            best_combined = selection_score
            best_epoch = epoch
            if save_path:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'val_clean_acc': val_clean_acc,
                    'val_adv_acc': val_adv_acc,
                    'phase': phase,
                    'combined_score': selection_score,
                }, save_path)

        if save_path:
            epoch_dir = save_path.replace('.pt', '_epochs')
            os.makedirs(epoch_dir, exist_ok=True)
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_clean_acc': val_clean_acc,
                'val_adv_acc': val_adv_acc,
                'best_val_acc': best_val_acc,
                'best_combined': best_combined,
                'best_epoch': best_epoch,
            }, f"{epoch_dir}/epoch_{epoch}.pt")

    if phase == 'A':
        print(f"  Best epoch: {best_epoch} | Best val clean acc: {best_val_acc:.4f}")
    else:
        print(f"  Best epoch: {best_epoch} | Best combined score: {best_combined:.4f} "
              f"(clean={best_val_acc:.4f}, adv tracked per-epoch)")

    if save_path and os.path.exists(save_path):
        print(f"  Reloading best weights from epoch {best_epoch} to pass to next phase...")
        ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])

    return model


# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic
# crash_test_greedy — evaluate clean + adversarial (k=1..4)
# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic

def crash_test_greedy(model, X_val, y_val, simulator, device=None,
                      k_values=None, label='', is_nlp=False, tokenizer=None, features=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if k_values is None:
        k_values = [1, 2, 3, 4]

    model.eval()
    model = model.to(device)

    n_eval = min(EVAL_SUBSAMPLE, len(X_val))
    X_eval = X_val[:n_eval]
    y_eval = y_val[:n_eval]

    clean_correct = 0
    with torch.no_grad():
        for i in range(0, n_eval, 1024):
            end = min(i + 1024, n_eval)
            X_b = torch.LongTensor(tokenizer.transform(X_eval[i:end], features)).to(device) if is_nlp else torch.FloatTensor(X_eval[i:end]).to(device)
            y_b = torch.LongTensor(y_eval[i:end]).to(device)
            clean_correct += (model(X_b).argmax(1) == y_b).sum().item()
    clean_acc = clean_correct / n_eval

    results = {'clean': clean_acc}

    print(f"  [Crash Test {label}] Clean={clean_acc:.4f}", end='')

    if simulator is not None:
        for k in k_values:
            adv_correct = 0
            with torch.no_grad():
                for i in range(0, n_eval, 1024):
                    end = min(i + 1024, n_eval)
                    X_adv = simulator.generate_greedy(X_eval[i:end], k=k)
                    X_adv_t = torch.LongTensor(tokenizer.transform(X_adv, features)).to(device) if is_nlp else torch.FloatTensor(X_adv).to(device)
                    y_b = torch.LongTensor(y_eval[i:end]).to(device)
                    adv_correct += (model(X_adv_t).argmax(1) == y_b).sum().item()

            adv_acc = adv_correct / n_eval
            results[f'adv_k{k}'] = adv_acc
            rr = adv_acc / max(clean_acc, 1e-8)
            print(f" | k={k}: {adv_acc:.4f}(RR={rr:.3f})", end='')

    print()
    return results


# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic
# run_sensitivity_analysis — run sensitivity_analysis_seq.py logic
# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic

def run_sensitivity_analysis(model, X_val, y_val, feature_names, num_classes,
                             n_continuous, save_csv_path, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"\n  Running sensitivity analysis...")
    model.eval()
    model = model.to(device)

    from src.adversarial.attacks import SensitivityAnalysis

    n_sens = min(5000, len(X_val))
    sens_indices = np.random.choice(len(X_val), n_sens, replace=False)
    X_sens = X_val[sens_indices].copy()
    y_sens = y_val[sens_indices].copy()

    sa = SensitivityAnalysis(
        X_sens, y_sens,
        feature_names if feature_names else [f'f{i}' for i in range(X_val.shape[2])],
        num_classes,
        n_continuous_features=n_continuous,
    )

    results = sa.analyze(model, X_sens, y_sens, device=device)

    rows = []
    for entry in results:
        rows.append({
            'feature': entry['feature'],
            'strategy': entry['strategy'],
            'drop': entry['drop'],
            'original_acc': entry.get('original_acc', 0) if 'original_acc' in entry else (entry['accuracy'] + entry['drop']),
            'perturbed_acc': entry['accuracy'],
        })

    df = pd.DataFrame(rows).sort_values('drop', ascending=False)
    os.makedirs(os.path.dirname(save_csv_path), exist_ok=True)
    df.to_csv(save_csv_path, index=False)
    print(f"  Sensitivity results saved to {save_csv_path}")
    print(f"  Top 5 vulnerable features:")
    for _, row in df.head(5).iterrows():
        print(f"    {row['feature']:<25} | {row['strategy']:<14} | drop={row['drop']*100:.1f}%")

    return df


# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic
# Discriminator and Router
# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic

class Discriminator(nn.Module):
    def __init__(self, input_size, seq_length, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h_cat = torch.cat([h[0], h[1]], dim=1)
        return self.head(h_cat).squeeze(1)

    def predict_proba(self, x):
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))

def train_discriminator(discriminator, X_train, simulator, device=None, epochs=25, batch_size=64, lr=1e-3, save_path='discriminator.pt'):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*65}\n  ENTRAÎNEMENT DU DISCRIMINATEUR\n{'='*65}")
    discriminator = discriminator.to(device)
    optimizer = torch.optim.AdamW(discriminator.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    n = len(X_train)
    best_acc = 0.0
    for epoch in range(1, epochs + 1):
        discriminator.train()
        n_half = n // 2
        idx = np.random.permutation(n)
        idx_clean = idx[:n_half]
        idx_adv = idx[n_half:n_half*2]
        k_values = np.random.randint(1, 4, size=n_half)
        X_adv_list = []
        for orig_i, k in zip(idx_adv, k_values):
            X_adv_list.append(simulator.generate_greedy(X_train[[orig_i]], k=k))
        X_adv_ep = np.concatenate(X_adv_list, axis=0)
        X_combined = np.concatenate([X_train[idx_clean], X_adv_ep], axis=0)
        labels_bin = np.array([0.0]*n_half + [1.0]*n_half, dtype=np.float32)
        perm = np.random.permutation(len(X_combined))
        X_combined = X_combined[perm]
        labels_bin = labels_bin[perm]
        loader = DataLoader(TensorDataset(torch.FloatTensor(X_combined), torch.FloatTensor(labels_bin)), batch_size=batch_size, shuffle=True)
        total_loss, total_correct, total_n = 0.0, 0, 0
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = discriminator(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(discriminator.parameters(), 1.0)
            optimizer.step()
            preds = (torch.sigmoid(logits) > 0.5).float()
            total_correct += (preds == yb).sum().item()
            total_loss += loss.item() * len(yb)
            total_n += len(yb)
        acc = total_correct / total_n
        print(f"  Epoch {epoch:3d}/{epochs}  Loss={total_loss/total_n:.4f}  Acc={acc:.4f}")
        if acc > best_acc:
            best_acc = acc
            torch.save({'model_state_dict': discriminator.state_dict(), 'accuracy': acc}, save_path)
    print(f"\n  Discriminateur — meilleure accuracy : {best_acc:.4f}\n  Sauvegardé → {save_path}")
    ckpt = torch.load(save_path, map_location=device)
    discriminator.load_state_dict(ckpt['model_state_dict'])
    return discriminator, best_acc

class IoTRouter(nn.Module):
    def __init__(self, normal_model, adversarial_model, discriminator, threshold=0.5, is_nlp=False, tokenizer=None, features=None):
        self.is_nlp = is_nlp
        self.tokenizer = tokenizer
        self.features = features
        super().__init__()
        self.normal = normal_model
        self.adversarial = adversarial_model
        self.discriminator = discriminator
        self.threshold = threshold

    @torch.no_grad()
    def predict(self, X):
        self.normal.eval()
        self.adversarial.eval()
        self.discriminator.eval()
        attack_scores = self.discriminator.predict_proba(X)
        is_attacked = (attack_scores >= self.threshold)
        logits_normal = self.normal(X)
        if hasattr(self, 'is_nlp') and self.is_nlp and self.tokenizer is not None:
            # Need X as numpy for tokenizer
            X_np = X.cpu().numpy()
            X_ids = self.tokenizer.transform(X_np, self.features)
            X_adj = torch.LongTensor(X_ids).to(X.device)
            logits_adv = self.adversarial(X_adj)
        else:
            logits_adv = self.adversarial(X)
        pred_normal = logits_normal.argmax(1)
        pred_adv = logits_adv.argmax(1)
        predictions = torch.where(is_attacked, pred_adv, pred_normal)
        routes = is_attacked.long()
        return predictions, routes, attack_scores

    def calibrate_threshold(self, X_clean, X_attacked, target_recall=0.95):
        with torch.no_grad():
            scores_clean = self.discriminator.predict_proba(X_clean).cpu().numpy()
            scores_attacked = self.discriminator.predict_proba(X_attacked).cpu().numpy()
        all_scores = np.concatenate([scores_clean, scores_attacked])
        all_labels = np.array([0]*len(scores_clean) + [1]*len(scores_attacked))
        thresholds = np.linspace(0.0, 1.0, 200)
        best_t, best_f1 = 0.5, 0.0
        for t in thresholds:
            preds = (all_scores >= t).astype(int)
            tp = ((preds == 1) & (all_labels == 1)).sum()
            fp = ((preds == 1) & (all_labels == 0)).sum()
            fn = ((preds == 0) & (all_labels == 1)).sum()
            recall = tp / (tp + fn + 1e-8)
            precision = tp / (tp + fp + 1e-8)
            f1 = 2 * precision * recall / (precision + recall + 1e-8)
            if recall >= target_recall and f1 > best_f1:
                best_f1 = f1
                best_t = t
        self.threshold = best_t
        print(f"  Seuil calibré : {best_t:.3f}  (recall attaques ≥ {target_recall:.0%})")
        return best_t

# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic
# train_model_greedy — orchestrate full 3-phase training for one model
# =====================================================================
# FIXED: nlp_cnn_bilstm_transformer properly uses embedded layer logic

def train_model_greedy(
    model_type, dataset_type, data_dict,
    batch_size=64, lr=5e-4,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    save_dir = f'{DRIVE_RESULTS_DIR}/models/{model_type}_greedy_{dataset_type}'
    os.makedirs(save_dir, exist_ok=True)

    X_train = data_dict['X_train']
    X_val = data_dict['X_val']
    X_test = data_dict['X_test']
    y_train = data_dict['y_train']
    y_val = data_dict['y_val']
    y_test = data_dict['y_test']
    features = data_dict.get('features', [])
    n_continuous = data_dict.get('n_continuous', X_train.shape[2])
    label_encoder = data_dict.get('label_encoder', None)

    input_size = X_train.shape[2]
    num_classes = len(np.unique(y_train))
    is_nlp = ('nlp' in model_type)
    tokenizer = None
    if is_nlp:
        tokenizer = create_tokenizer()
        print(f"\n  [TOKENIZER] Fitting BPE tokenizer on training data...")
        tokenizer.fit(X_train, features, verbose=False)

    print(f"\n{'#'*80}")
    print(f"  GREEDY ADVERSARIAL TRAINING — {model_type.upper()} on {dataset_type.upper()}")
    print(f"{'#'*80}")
    print(f"  Input size: {input_size} | Classes: {num_classes}")
    print(f"  Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
    print(f"  Save dir: {save_dir}")

    all_crash_results = {}

    # ─── Modele antagoniste from-scratch (aucun poids Phase A chargé) ─────────────
    # Le modèle antagoniste est spécialisé uniquement contre les attaques.
    # Charger les poids sains (Phase A) introduit un biais vers les features vulnérables.
    # => On part de poids aléatoires pour forcer l'apprentissage des features robustes.
    phase_a_path = f'{save_dir}/phase_a_model.pt'
    sens_csv_path = f'{save_dir}/sensitivity_results.csv'

    model = create_model(model_type, input_size, num_classes)
    model = model.to(device)
    print(f"\n  [FROM-SCRATCH] Modèle antagoniste initialisé avec des poids aléatoires.")
    print(f"  (Phase A ignorée — aucun checkpoint sain chargé)")

    # ─── Sensitivity Analysis (after Phase A) ─────────────────────────────
    feature_names = features if features else [f'f{i}' for i in range(input_size)]

    if os.path.exists(sens_csv_path):
        print(f"\n  Sensitivity results found in Drive. Loading...")
        sensitivity = load_sensitivity_results(sens_csv_path, feature_names)
    else:
        print(f"\n  [Simulator] Loading clean model to compute accurate vulnerability dictionary...")
        temp_model = create_model(model_type, input_size, num_classes).to(device)
        clean_model_path = f'{DRIVE_RESULTS_DIR}/models/{model_type}_{dataset_type}/best_val_model.pt'
        if not os.path.exists(clean_model_path):
            clean_model_path = phase_a_path
        try:
            ckpt_a = torch.load(clean_model_path, map_location=device)
            temp_model.load_state_dict(ckpt_a.get('model_state_dict', ckpt_a))
            print(f"  Clean weights loaded successfully from {clean_model_path} for sensitivity analysis.")
        except Exception as e:
            print(f"  WARN: Failed to load clean model for sensitivity ({e}). Using random weights.")
        run_sensitivity_analysis(
            temp_model, X_val, y_val, feature_names, num_classes,
            n_continuous, sens_csv_path, device=device,
        )
        del temp_model
        sensitivity = load_sensitivity_results(sens_csv_path, feature_names)

    feature_stats = GreedyAttackSimulator.compute_feature_stats(X_train)
    simulator = GreedyAttackSimulator(sensitivity, feature_stats,
                                       feature_names=feature_names, n_continuous=n_continuous)
    simulator.save_dictionary(f'{save_dir}/vulnerability_dictionary.json', feature_names)

    # ─── PHASE B (epochs 16-30): 30% adversarial, k_max=2 ─────────────────
    phase_b_path = f'{save_dir}/phase_b_model.pt'

    needs_train_b = not os.path.exists(phase_b_path)
    if not needs_train_b:
        if not os.path.exists(f"{phase_b_path.replace('.pt', '_epochs')}/epoch_{PHASE_B_EPOCHS}.pt"):
            needs_train_b = True

    if not needs_train_b:
        print(f"\n  Phase B model found in Drive. Loading...")
        ckpt = torch.load(phase_b_path, map_location=device)
        try:
            model.load_state_dict(ckpt['model_state_dict'])
            model = model.to(device)
            print(f"  Loaded Phase B model (epoch {ckpt.get('epoch', '?')}, "
                  f"clean_acc={ckpt.get('val_clean_acc', 0):.4f}, "
                  f"adv_acc={ckpt.get('val_adv_acc', 0):.4f})")
        except RuntimeError as e:
            if 'mismatch' in str(e):
                print(f"  Size mismatch: IGNORING Phase B checkpoint. Retraining...")
                needs_train_b = True
            else:
                raise e

    if needs_train_b:
        model = train_greedy_phase(
            model, X_train, y_train, X_val, y_val,
            phase='B', start_epoch=PHASE_A_EPOCHS + 1, end_epoch=PHASE_B_EPOCHS,
            mix_ratio=PHASE_B_MIX_RATIO, k_max=PHASE_B_K_MAX,
            p_drop=0.1, sigma_noise=0.01, afd_lambda=0.0,
            simulator=simulator, device=device, lr=lr,
            batch_size=batch_size, save_path=phase_b_path, is_nlp=is_nlp, tokenizer=tokenizer, features=features,
            adv_method='stratified',  # Méthode 3 — exhaustif k=1 (100% couverture single-feature)
        )

    ct_b = crash_test_greedy(model, X_val, y_val, simulator=simulator, device=device, label='Phase B', is_nlp=is_nlp, tokenizer=tokenizer, features=features)
    all_crash_results['phase_b'] = ct_b

    # NOTE: Simulateur fixe pour toutes les phases (B, C, D).
    # On n'effectue PAS de nouvelle sensitivity analysis entre les phases.
    # Raison: le curriculum augmente la difficulte des attaques (k=1->k=2->k=3,4),
    # pas les features ciblees. Changer de simulator cause du catastrophic forgetting.
    simulator_c = simulator  # meme simulateur que Phase B

    # ─── PHASE C (epochs 31-50): 70% adversarial, k_max=4 ─────────────────
    phase_c_path = f'{save_dir}/phase_c_model.pt'

    needs_train_c = not os.path.exists(phase_c_path)
    if not needs_train_c:
        if not os.path.exists(f"{phase_c_path.replace('.pt', '_epochs')}/epoch_{PHASE_C_EPOCHS}.pt"):
            needs_train_c = True

    if not needs_train_c:
        print(f"\n  Phase C model found in Drive. Loading...")
        ckpt = torch.load(phase_c_path, map_location=device)
        try:
            model.load_state_dict(ckpt['model_state_dict'])
            model = model.to(device)
            print(f"  Loaded Phase C model (epoch {ckpt.get('epoch', '?')}, "
                  f"clean_acc={ckpt.get('val_clean_acc', 0):.4f}, "
                  f"adv_acc={ckpt.get('val_adv_acc', 0):.4f})")
        except RuntimeError as e:
            if 'mismatch' in str(e):
                print(f"  Size mismatch: IGNORING Phase C checkpoint. Retraining...")
                needs_train_c = True
            else:
                raise e

    if needs_train_c:
        model = train_greedy_phase(
            model, X_train, y_train, X_val, y_val,
            phase='C', start_epoch=PHASE_B_EPOCHS + 1, end_epoch=PHASE_C_EPOCHS,
            mix_ratio=PHASE_C_MIX_RATIO, k_max=PHASE_C_K_MAX,
            p_drop=0.2, sigma_noise=0.01, afd_lambda=0.0,
            simulator=simulator_c, device=device, lr=lr,
            batch_size=batch_size, save_path=phase_c_path, is_nlp=is_nlp, tokenizer=tokenizer, features=features,
            adv_method='stratified',
            top_n_k2=12,             # 12 features pour paires k=2 (au lieu de 8)
        )

    ct_c = crash_test_greedy(model, X_val, y_val, simulator=simulator, device=device, label='Phase C', is_nlp=is_nlp, tokenizer=tokenizer, features=features)
    all_crash_results['phase_c'] = ct_c

    simulator_d = simulator  # meme simulateur que Phase B et C

    # ─── PHASE D (epochs 51-65): 95% adversarial, k_max=4 ─────────────────
    phase_d_path = f'{save_dir}/phase_d_model.pt'

    needs_train_d = not os.path.exists(phase_d_path)
    if not needs_train_d:
        if not os.path.exists(f"{phase_d_path.replace('.pt', '_epochs')}/epoch_{PHASE_D_EPOCHS}.pt"):
            needs_train_d = True

    if not needs_train_d:
        print(f"\n  Phase D model found in Drive. Loading...")
        ckpt = torch.load(phase_d_path, map_location=device)
        try:
            model.load_state_dict(ckpt['model_state_dict'])
            model = model.to(device)
            print(f"  Loaded Phase D model (epoch {ckpt.get('epoch', '?')}, "
                  f"clean_acc={ckpt.get('val_clean_acc', 0):.4f}, "
                  f"adv_acc={ckpt.get('val_adv_acc', 0):.4f})")
        except RuntimeError as e:
            if 'mismatch' in str(e):
                print(f"  Size mismatch: IGNORING Phase D checkpoint. Retraining...")
                needs_train_d = True
            else:
                raise e

    if needs_train_d:
        model = train_greedy_phase(
            model, X_train, y_train, X_val, y_val,
            phase='D', start_epoch=PHASE_C_EPOCHS + 1, end_epoch=PHASE_D_EPOCHS,
            mix_ratio=PHASE_D_MIX_RATIO, k_max=PHASE_D_K_MAX,
            p_drop=0.2, sigma_noise=0.01, afd_lambda=0.0,
            simulator=simulator_d, device=device, lr=lr,
            batch_size=batch_size, save_path=phase_d_path, is_nlp=is_nlp, tokenizer=tokenizer, features=features,
            adv_method='stratified',  # Méthode 3+2 — exhaustif k=1+k=2 + worst-case k=3,4 (PGD-like)
            n_candidates=16,          # N candidats pour la sélection worst-case (augmenté)
            top_n_k2=8,               # Top-8 features pour les paires k=2
        )

    ct_d = crash_test_greedy(model, X_val, y_val, simulator=simulator_d, device=device, label='Phase D', is_nlp=is_nlp, tokenizer=tokenizer, features=features)
    all_crash_results['phase_d'] = ct_d

    # ─── PHASE E: Discriminator ──────────────────────────────────────────
    disc_path = f'{save_dir}/discriminator.pt'
    disc = Discriminator(input_size=input_size, seq_length=10, hidden_size=64)
    if os.path.exists(disc_path):
        print(f"\n  Discriminator model found in Drive. Loading...")
        ckpt = torch.load(disc_path, map_location=device)
        disc.load_state_dict(ckpt['model_state_dict'])
        disc_acc = ckpt.get('accuracy', 0.95)
        disc = disc.to(device)
        print(f"  Loaded Discriminator (acc={disc_acc:.4f})")
    else:
        disc, disc_acc = train_discriminator(
            discriminator=disc,
            X_train=X_train,
            simulator=simulator,
            device=device,
            epochs=25,
            batch_size=batch_size,
            save_path=disc_path
        )

    # ─── Final evaluation on test set with Router ────────────────────────
    print(f"\n{'='*80}")
    print(f"  RÉSULTATS ATTENDUS APRÈS ENTRAÎNEMENT")
    print(f"{'='*80}")
    model.eval() # Adversarial model
    disc.eval()

    # Create & load normal model
    normal_model_path = f'{DRIVE_RESULTS_DIR}/models/{model_type}_{dataset_type}/best_val_model.pt'
    if not os.path.exists(normal_model_path):
        normal_model_path = phase_a_path
    normal_model = create_model(model_type, input_size, num_classes)
    if os.path.exists(normal_model_path):
        try:
            ckpt_nor = torch.load(normal_model_path, map_location=device)
            if 'model_state_dict' in ckpt_nor:
                normal_model.load_state_dict(ckpt_nor['model_state_dict'])
            else:
                normal_model.load_state_dict(ckpt_nor)
            print(f"  Normal model loaded from {normal_model_path}")
        except Exception as e:
            print(f"  Could not load normal model: {e}. Using random weights.")
    else:
        print(f"  WARN: Normal model not found at {normal_model_path}. Using random weights.")
    normal_model = normal_model.to(device)
    normal_model.eval()

    router = IoTRouter(normal_model, model, disc, threshold=0.5, is_nlp=is_nlp, tokenizer=tokenizer, features=features)

    # Calibrate router limit
    X_val_clean_sub = torch.FloatTensor(X_val[:min(len(X_val), 1000)]).to(device)
    X_val_adv_sub = torch.FloatTensor(simulator.generate_greedy(X_val[:min(len(X_val), 1000)], k=4)).to(device)
    router.calibrate_threshold(X_val_clean_sub, X_val_adv_sub, target_recall=0.95)

    # Evaluate Clean
    test_dataset = IoTSequenceDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE)
    correct_clean = 0
    total = 0
    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            pred, _, _ = router.predict(X_b)
            total += y_b.size(0)
            correct_clean += pred.eq(y_b).sum().item()
    clean_acc = correct_clean / total

    # Evaluate pure models
    correct_normal_clean = 0
    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            _, pred = normal_model(X_b).max(1)
            correct_normal_clean += pred.eq(y_b).sum().item()
    normal_clean_acc = correct_normal_clean / max(total, 1)

    # Evaluate Adv k=4 on Adv Model
    adv_results = {}
    correct_adv_model_k4 = 0
    correct_router_adv_k4 = 0
    total_adv = 0

    for k in [1, 2, 3, 4]:
        c_adv_k = 0
        t_adv_k = 0
        for i in range(0, min(len(X_test), EVAL_SUBSAMPLE), EVAL_BATCH_SIZE):
            batch_end = min(i + EVAL_BATCH_SIZE, len(X_test), EVAL_SUBSAMPLE)
            X_sub = X_test[i:batch_end]
            y_sub = y_test[i:batch_end]

            X_adv = simulator.generate_greedy(X_sub, k=k)
            X_adv_t = torch.LongTensor(tokenizer.transform(X_adv, features)).to(device) if is_nlp else torch.FloatTensor(X_adv).to(device)
            y_sub_t = torch.LongTensor(y_sub).to(device)

            with torch.no_grad():
                _, pred_adv = model(X_adv_t).max(1)

            c_adv_k += pred_adv.eq(y_sub_t).sum().item()
            t_adv_k += len(y_sub)
        adv_results[f'k{k}'] = c_adv_k / max(t_adv_k, 1)
        if k == 4:
            adv_model_k4_acc = adv_results[f'k{k}']
            # router test on k4
            for i in range(0, min(len(X_test), EVAL_SUBSAMPLE), EVAL_BATCH_SIZE):
                batch_end = min(i + EVAL_BATCH_SIZE, len(X_test), EVAL_SUBSAMPLE)
                X_sub = X_test[i:batch_end]
                y_sub = y_test[i:batch_end]
                X_adv = simulator.generate_greedy(X_sub, k=4)
                X_adv_t = torch.LongTensor(tokenizer.transform(X_adv, features)).to(device) if is_nlp else torch.FloatTensor(X_adv).to(device)
                y_sub_t = torch.LongTensor(y_sub).to(device)
                with torch.no_grad():
                    pred_router, _, _ = router.predict(X_adv_t)
                correct_router_adv_k4 += pred_router.eq(y_sub_t).sum().item()
            router_k4_acc = correct_router_adv_k4 / max(t_adv_k, 1)

    global_acc = (clean_acc + router_k4_acc) / 2.0

    print(f"──────────────────────────────────────")
    print(f"    Modèle normal     → Clean accuracy     : {normal_clean_acc*100:.1f}%")
    print(f"    Modèle antagoniste→ Adversarial acc k4 : {adv_model_k4_acc*100:.1f}%")
    print(f"    Discriminateur    → Détection attaque  : {disc_acc*100:.1f}%")
    print(f"    Système complet   → Accuracy globale   : {global_acc*100:.1f}%")
    print(f"================================================================================")

    results = {
        'model_type': model_type,
        'dataset_type': dataset_type,
        'clean_accuracy': clean_acc,
        'adversarial_accuracies': adv_results,
        'crash_tests': all_crash_results,
        'input_size': input_size,
        'num_classes': num_classes,
        'phases': {
            'A': {'epochs': f'1-{PHASE_A_EPOCHS}', 'mix_ratio': PHASE_A_MIX_RATIO, 'k_max': 0},
            'B': {'epochs': f'{PHASE_A_EPOCHS+1}-{PHASE_B_EPOCHS}', 'mix_ratio': PHASE_B_MIX_RATIO, 'k_max': PHASE_B_K_MAX},
            'C': {'epochs': f'{PHASE_B_EPOCHS+1}-{PHASE_C_EPOCHS}', 'mix_ratio': PHASE_C_MIX_RATIO, 'k_max': PHASE_C_K_MAX},
            'D': {'epochs': f'{PHASE_C_EPOCHS+1}-{PHASE_D_EPOCHS}', 'mix_ratio': PHASE_D_MIX_RATIO, 'k_max': PHASE_D_K_MAX},
        }
    }

    with open(f'{save_dir}/greedy_results.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)

    aggressive_cleanup()
    return results


print('Greedy adversarial training functions loaded.')





# Phase 4 : Greedy Adversarial Training on CSV

In [ ]:
# ─── MODEL: LSTM on CSV (Greedy Adversarial) ────────────────────────
MODEL = 'lstm'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — LSTM on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_csv')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n LSTM on CSV DONE')

In [ ]:
# ─── MODEL: BiLSTM on CSV (Greedy Adversarial) ────────────────────────
MODEL = 'bilstm'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — BILSTM on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_csv')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n BILSTM on CSV DONE')

In [ ]:
# ─── MODEL: CNN-LSTM on CSV (Greedy Adversarial) ────────────────────────
MODEL = 'cnn_lstm'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — CNN-LSTM on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_csv')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n CNN-LSTM on CSV DONE')

In [ ]:
# ─── MODEL: XGBoost-LSTM on CSV (Greedy Adversarial) ────────────────────────
MODEL = 'xgboost_lstm'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — XGBOOST-LSTM on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_csv')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n XGBOOST-LSTM on CSV DONE')

In [ ]:
# ─── MODEL: Transformer on CSV (Greedy Adversarial) ────────────────────────
MODEL = 'transformer'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — TRANSFORMER on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_csv')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n TRANSFORMER on CSV DONE')

In [ ]:
# ─── MODEL: CNN-BiLSTM-Transformer on CSV (Greedy Adversarial) ────────────────────────
MODEL = 'cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — CNN-BILSTM-TRANSFORMER on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_csv')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n CNN-BILSTM-TRANSFORMER on CSV DONE')

In [ ]:
# ─── MODEL: NLP-CNN-BiLSTM-Transformer on CSV (Greedy Adversarial) ────────────────────────
MODEL = 'nlp_cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — NLP-CNN-BILSTM-TRANSFORMER on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_csv')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n NLP-CNN-BILSTM-TRANSFORMER on CSV DONE')

In [ ]:
# ─── MODEL: NLP-Transformer on CSV (Greedy Adversarial) ────────────────────────
MODEL = 'nlp_transformer'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — NLP-TRANSFORMER on CSV')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_csv')

if DATASETS in ['csv', 'both']:
    data = load_dataset_from_drive('csv')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='csv',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_csv')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load CSV data')
else:
    print('Skipping CSV dataset')

print(f'\n NLP-TRANSFORMER on CSV DONE')

# RAM Cleanup Before JSON Phase

In [ ]:
# ─── CLEANUP RAM BEFORE JSON PHASE ──────────────────────────────────────
print('\n' + '='*80)
print('  CLEANING RAM BEFORE JSON PHASE...')
print('='*80 + '\n')

aggressive_cleanup()
print('\n RAM cleaned. Ready for JSON phase.')

# Phase 5 : Greedy Adversarial Training on JSON

In [ ]:
def load_and_display_json_dataset(json_data_dir, seq_length=10, stride=10, max_records=None, save_dir=None):
    """Charge COMPLÈTEMENT le dataset JSON, affiche les infos, et sauvegarde sur Drive."""
    import sys
    import gc
    import numpy as np
    import pickle

    sys.path.insert(0, '/content/pfe')

    print("\n" + "=" * 70)
    print("  CHARGEMENT COMPLET DU DATASET JSON")
    print("=" * 70)

    print(f"\n  Répertoire : {json_data_dir}")
    print(f"  Seq length : {seq_length} | Stride : {stride}")
    print(f"  Max records: {max_records if max_records else 'Tous'}")

    json_files = sorted(glob.glob(f'{json_data_dir}/**/*.json', recursive=True))
    print(f"\n  Fichiers JSON trouvés : {len(json_files)}")

    for f in json_files:
        size_gb = os.path.getsize(f) / (1024**3)
        print(f"    {os.path.basename(f):<30s} : {size_gb:>10.2f} GB")

    total_gb = sum(os.path.getsize(f) for f in json_files) / (1024**3)
    print(f"\n  Taille totale : {total_gb:.2f} GB")

    # Charger le dataset complet via le vrai pipeline JSON
    print("\n  Chargement via le pipeline JSON (JsonIoTDataProcessor)...")
    from src.data.json_preprocessor import JsonIoTDataProcessor

    processor = JsonIoTDataProcessor()
    result = processor.process_all(
        data_dir=json_data_dir,
        seq_length=seq_length,
        stride=stride,
        max_records=max_records,
    )

    X_train, X_val, X_test, y_train, y_val, y_test, features, scaler, label_encoder = result
    n_continuous = 36  # JSON pipeline features

    print(f"\n  {'='*70}")
    print(f"  RÉSULTAT DU CHARGEMENT")
    print(f"  {'='*70}")
    print(f"    Features ({len(features)}) : {features[:5]}...")
    print(f"    Classes ({len(label_encoder.classes_)}) : {list(label_encoder.classes_)}")
    print(f"\n  Shapes des séquences (seq_length={seq_length}, stride={stride}) :")
    print(f"    Train : {X_train.shape}  →  {len(X_train):,} séquences")
    print(f"    Val   : {X_val.shape}  →  {len(X_val):,} séquences")
    print(f"    Test  : {X_test.shape}  →  {len(X_test):,} séquences")
    print(f"    Total : {len(X_train) + len(X_val) + len(X_test):,} séquences")

    print(f"\n  Distribution des classes (train) :")
    for cls in label_encoder.classes_:
        cls_id = label_encoder.transform([cls])[0]
        count = int(np.sum(y_train == cls_id))
        bar = '█' * max(1, count // 50)
        print(f"    {cls:<30s} : {count:>6,}  {bar}")

    # ─── Sauvegarder sur Drive ────────────────────────────────────────────
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        print(f"\n  💾 Sauvegarde du dataset JSON pré-traité sur Drive...")
        print(f"     Répertoire : {save_dir}")

        np.save(f'{save_dir}/X_train.npy', X_train)
        np.save(f'{save_dir}/X_val.npy', X_val)
        np.save(f'{save_dir}/X_test.npy', X_test)
        np.save(f'{save_dir}/y_train.npy', y_train)
        np.save(f'{save_dir}/y_val.npy', y_val)
        np.save(f'{save_dir}/y_test.npy', y_test)

        with open(f'{save_dir}/json_metadata.pkl', 'wb') as f:
            pickle.dump({
                'features': features,
                'scaler': scaler,
                'label_encoder': label_encoder,
                'n_continuous': n_continuous,
                'seq_length': seq_length,
                'stride': stride,
            }, f)

        # Marker file to indicate preprocessing is complete
        with open(f'{save_dir}/json_ready', 'w') as f:
            f.write('ready')

        saved_gb = (X_train.nbytes + X_val.nbytes + X_test.nbytes +
                    y_train.nbytes + y_val.nbytes + y_test.nbytes) / (1024**3)
        print(f"  ✅ Dataset JSON sauvegardé ({saved_gb:.2f} GB)")
        print(f"     Fichiers : X_train, X_val, X_test, y_train, y_val, y_test, json_metadata.pkl")

    print(f"\n  {'='*70}")
    print(f"  ✅ Dataset JSON chargé complètement en RAM")
    print(f"  {'='*70}\n")

    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'features': features, 'scaler': scaler,
        'label_encoder': label_encoder, 'n_continuous': n_continuous
    }

# ─── JSON preprocessing directory on Drive ─────────────────────────────
JSON_PREPROCESSED_DIR = f'{DRIVE_RESULTS_DIR}/preprocessed/json'

if DATASETS in ['json', 'both']:
    json_data = load_and_display_json_dataset(
        JSON_DATA_DIR, seq_length=SEQ_LENGTH, stride=STRIDE,
        max_records=MAX_RECORDS, save_dir=JSON_PREPROCESSED_DIR
    )
else:
    json_data = None
    print('Skipping JSON dataset loading — DATASETS is not json or both')


In [ ]:
# ─── Cell: Load JSON Dataset ─────────────────────────────────────────────
if DATASETS in ['json', 'both']:
    json_data = load_dataset_from_drive('json')
else:
    json_data = None
    print('Skipping JSON dataset')

In [ ]:
# ─── MODEL: LSTM on JSON (Greedy Adversarial) ────────────────────────
MODEL = 'lstm'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — LSTM on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_json')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n LSTM on JSON DONE')

In [ ]:
# ─── MODEL: BiLSTM on JSON (Greedy Adversarial) ────────────────────────
MODEL = 'bilstm'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — BILSTM on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_json')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n BILSTM on JSON DONE')

In [ ]:
# ─── MODEL: CNN-LSTM on JSON (Greedy Adversarial) ────────────────────────
MODEL = 'cnn_lstm'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — CNN-LSTM on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_json')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n CNN-LSTM on JSON DONE')

In [ ]:
# ─── MODEL: XGBoost-LSTM on JSON (Greedy Adversarial) ────────────────────────
MODEL = 'xgboost_lstm'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — XGBOOST-LSTM on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_json')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n XGBOOST-LSTM on JSON DONE')

In [ ]:
# ─── MODEL: Transformer on JSON (Greedy Adversarial) ────────────────────────
MODEL = 'transformer'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — TRANSFORMER on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_json')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n TRANSFORMER on JSON DONE')

In [ ]:
# ─── MODEL: CNN-BiLSTM-Transformer on JSON (Greedy Adversarial) ────────────────────────
MODEL = 'cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — CNN-BILSTM-TRANSFORMER on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_json')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n CNN-BILSTM-TRANSFORMER on JSON DONE')

In [ ]:
# ─── MODEL: NLP-CNN-BiLSTM-Transformer on JSON (Greedy Adversarial) ────────────────────────
MODEL = 'nlp_cnn_bilstm_transformer'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — NLP-CNN-BILSTM-TRANSFORMER on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_json')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n NLP-CNN-BILSTM-TRANSFORMER on JSON DONE')

In [ ]:
# ─── MODEL: NLP-Transformer on JSON (Greedy Adversarial) ────────────────────────
MODEL = 'nlp_transformer'
print(f'\n{"#"*80}')
print(f'  GREEDY ADVERSARIAL — NLP-TRANSFORMER on JSON')
print(f'{"#"*80}\n')

log_memory(f'before_{MODEL}_json')

if DATASETS in ['json', 'both']:
    data = load_dataset_from_drive('json')
    if data is not None:
        results = train_model_greedy(
            model_type=MODEL,
            dataset_type='json',
            data_dict=data,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
        )
        log_memory(f'after_{MODEL}_json')
        del data  # release numpy arrays from RAM
        try:
            del results
        except Exception:
            pass
        aggressive_cleanup()
    else:
        print('Failed to load JSON data')
else:
    print('Skipping JSON dataset')

print(f'\n NLP-TRANSFORMER on JSON DONE')

# Phase 6 : Results Visualization & Comparison

In [ ]:
# ─── Cell: Visualize Results ─────────────────────────────────────────────
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path(DRIVE_RESULTS_DIR) / 'models'
if not results_dir.exists():
    print('No results found yet — run the training cells first.')
else:
    greedy_dirs = sorted([d for d in results_dir.iterdir() if d.is_dir() and 'greedy' in d.name])

    if not greedy_dirs:
        print('No greedy results found yet.')
    else:
        fig, axes = plt.subplots(1, 2, figsize=(20, 8))

        for idx, dataset in enumerate(['csv', 'json']):
            ax = axes[idx]
            ds_dirs = [d for d in greedy_dirs if dataset in d.name]
            models = []
            clean_accs = []
            adv_k1 = []
            adv_k2 = []
            adv_k3 = []
            adv_k4 = []

            for d in sorted(ds_dirs):
                rf = d / 'greedy_results.json'
                if not rf.exists():
                    continue
                with open(rf) as f:
                    res = json.load(f)

                model_name = d.name.replace(f'_greedy_{dataset}', '').upper()
                models.append(model_name)
                clean_accs.append(res.get('clean_accuracy', 0))
                adv_k1.append(res.get('adversarial_accuracies', {}).get('k1', 0))
                adv_k2.append(res.get('adversarial_accuracies', {}).get('k2', 0))
                adv_k3.append(res.get('adversarial_accuracies', {}).get('k3', 0))
                adv_k4.append(res.get('adversarial_accuracies', {}).get('k4', 0))

            if not models:
                ax.set_title(f'{dataset.upper()} — No results yet')
                continue

            x = np.arange(len(models))
            width = 0.15

            ax.bar(x - 2*width, clean_accs, width, label='Clean', color='#2ecc71')
            ax.bar(x - width, adv_k1, width, label='Adv k=1', color='#3498db')
            ax.bar(x, adv_k2, width, label='Adv k=2', color='#e67e22')
            ax.bar(x + width, adv_k3, width, label='Adv k=3', color='#e74c3c')
            ax.bar(x + 2*width, adv_k4, width, label='Adv k=4', color='#9b59b6')

            ax.set_ylabel('Accuracy')
            ax.set_title(f'{dataset.upper()} — Greedy Adversarial Results')
            ax.set_xticks(x)
            ax.set_xticklabels(models, rotation=45, ha='right')
            ax.legend()
            ax.set_ylim(0, 1.05)
            ax.grid(axis='y', alpha=0.3)

        plt.tight_layout()
        save_path = f'{DRIVE_RESULTS_DIR}/greedy_comparison.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Plot saved to {save_path}')

        # Phase progression plot per model
        for d in greedy_dirs:
            rf = d / 'greedy_results.json'
            if not rf.exists():
                continue
            with open(rf) as f:
                res = json.load(f)

            ct = res.get('crash_tests', {})
            if not ct:
                continue

            fig, ax = plt.subplots(figsize=(8, 5))
            phases = sorted(ct.keys())
            clean_vals = [ct[p].get('clean', 0) for p in phases]
            ax.plot(phases, clean_vals, 'o-', label='Clean', color='#2ecc71', linewidth=2)
            for k in [1, 2, 3, 4]:
                k_key = f'adv_k{k}'
                vals = [ct[p].get(k_key, None) for p in phases]
                if any(v is not None for v in vals):
                    vals = [v if v is not None else 0 for v in vals]
                    ax.plot(phases, vals, 'o--', label=f'Adv k={k}', alpha=0.7)

            ax.set_title(f"{d.name} — Accuracy by Phase")
            ax.set_xlabel('Phase')
            ax.set_ylabel('Accuracy')
            ax.legend()
            ax.grid(alpha=0.3)
            ax.set_ylim(0, 1.05)
            plt.tight_layout()
            plt.show()

In [ ]:
# ─── Cell: Comparative Summary Table ─────────────────────────────────────
import json
from pathlib import Path

results_dir = Path(DRIVE_RESULTS_DIR) / 'models'
if not results_dir.exists():
    print('No results found.')
else:
    greedy_dirs = sorted([d for d in results_dir.iterdir() if d.is_dir() and 'greedy' in d.name])

    print(f"{'Model + Dataset':<40} {'Clean':>8} {'k=1':>8} {'k=2':>8} {'k=3':>8} {'k=4':>8}")
    print('-' * 80)

    for d in greedy_dirs:
        rf = d / 'greedy_results.json'
        if not rf.exists():
            continue
        with open(rf) as f:
            res = json.load(f)

        name = d.name
        clean = res.get('clean_accuracy', 0)
        adv = res.get('adversarial_accuracies', {})
        k1 = adv.get('k1', 0)
        k2 = adv.get('k2', 0)
        k3 = adv.get('k3', 0)
        k4 = adv.get('k4', 0)

        print(f"{name:<40} {clean:>8.4f} {k1:>8.4f} {k2:>8.4f} {k3:>8.4f} {k4:>8.4f}")

    print('-' * 80)

    # Crash test summary
    print(f"\n{'Model + Dataset':<40} {'Phase':>8} {'Clean':>8} {'k=1':>8} {'k=2':>8} {'k=3':>8} {'k=4':>8}")
    print('-' * 100)
    for d in greedy_dirs:
        rf = d / 'greedy_results.json'
        if not rf.exists():
            continue
        with open(rf) as f:
            res = json.load(f)

        name = d.name
        ct = res.get('crash_tests', {})
        for phase_key in sorted(ct.keys()):
            phase_data = ct[phase_key]
            clean = phase_data.get('clean', 0)
            k1 = phase_data.get('adv_k1', 0)
            k2 = phase_data.get('adv_k2', 0)
            k3 = phase_data.get('adv_k3', 0)
            k4 = phase_data.get('adv_k4', 0)
            print(f"{name:<40} {phase_key:>8} {clean:>8.4f} {k1:>8.4f} {k2:>8.4f} {k3:>8.4f} {k4:>8.4f}")
    print('-' * 100)
    print('\n Comparison complete.')

In [ ]:
# ─── Git Push ──────────────────────────────────────────────────────────────
import subprocess

print('Pushing to GitHub...')
result = subprocess.run(['git', 'add', '-A'], capture_output=True, text=True, cwd='/content/pfe')
print(f'  git add: {result.returncode}')

result = subprocess.run(['git', 'commit', '-m', 'Update greedy adversarial training results'], capture_output=True, text=True, cwd='/content/pfe')
if result.returncode == 0:
    print(f'  git commit: OK')
else:
    print(f'  git commit: {result.stdout.strip()} {result.stderr.strip()}')

result = subprocess.run(['git', 'push'], capture_output=True, text=True, cwd='/content/pfe')
if result.returncode == 0:
    print('  Push successful!')
else:
    print(f'  git push stderr: {result.stderr.strip()[:500]}')